# Fase 2 - Analisis inferencial


In [1]:
#Importe de librerias
import pandas as pd
import scipy.stats as stats
from scipy.stats import chi2_contingency
import numpy as np
#pip install scikit_posthocs
import scikit_posthocs as sp
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.multitest import multipletests





In [2]:
datos = pd.read_csv('https://github.com/rodrigo-elias-01/Data-Science-Grupo15/raw/refs/heads/main/data/EPHC_2025_limpio.csv')
datos.head()

,UPM,NVIVI,NHOGA,L02,DPTO,NOMBRE_DPTO,P02,grupo_etario,añoest,ED0504,nivel_educ_simplificado,ED06C,ED08,asiste_actualmente,ED09,ED10
0,14,13,1,1,0,Asunción,65,65+,15,2403,Terciaria/Universitaria,14.0,NaN,NaN,NaN,NaN
1,14,13,1,2,0,Asunción,69,65+,18,2406,Terciaria/Universitaria,10.0,NaN,NaN,NaN,NaN
2,14,15,1,1,0,Asunción,78,65+,6,306,Primaria (EEB 1-6),NaN,NaN,NaN,NaN,NaN
3,14,15,1,2,0,Asunción,78,65+,12,606,Secundaria/Media,NaN,NaN,NaN,NaN,NaN
4,14,16,1,1,0,Asunción,40,35-49,17,2405,Terciaria/Universitaria,1.0,19.0,0.0,NaN,2.0


## Contraste 1: Años de estudio por departamento
Ho: Las distribuciones son iguales en los 16 departamentos

H1: Al menos una difiere

In [3]:
# Datos agrupados por departamentos
grupos = {name: g["añoest"].dropna().to_numpy()
          for name, g in datos.groupby("NOMBRE_DPTO")}
nombres = list(grupos)
k = len(grupos)
n = sum(len(v) for v in grupos.values())

shapiro_filas = []
for name, vals in grupos.items():
    if len(vals) >= 3:
        stat_shapiro, p = stats.shapiro(vals)
    else:
        stat_shapiro, p = np.nan, np.nan
    shapiro_filas.append([name, len(vals), np.mean(vals), np.median(vals),
                         np.std(vals, ddof=1), stat_shapiro, p])

# Normalidad por grupo (Shapiro-Wilk)
shapiro_datos = pd.DataFrame(
    shapiro_filas,
    columns=["Departamento", "n", "Media", "Mediana", "DE", "stat_shapiro", "p valor"]
)
shapiro_datos["Cumple normalidad (α=0.05)"] = shapiro_datos["p valor"] > 0.05

print("Shapiro-Wilk por departamento:")
display(shapiro_datos)
print(f"\nDepartamentos que NO cumplen normalidad: "
      f"{(~shapiro_datos['Cumple normalidad (α=0.05)']).sum()} / {k}")

# Prueba de Levene
lev = stats.levene(*grupos.values(), center="median")
print(f"\nLevene: W={lev.statistic:.3f}; p={lev.pvalue:.4g}")


Shapiro-Wilk por departamento:


/home/hugo/.venvs/data-science/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:579: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 6113.
  res = hypotest_fun_out(*samples, **kwds)
/home/hugo/.venvs/data-science/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:579: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 9153.
  res = hypotest_fun_out(*samples, **kwds)


,Departamento,n,Media,Mediana,DE,stat_shapiro,p valor,Cumple normalidad (α=0.05)
0,Alto Paraná,6113,8.282022,8.0,4.940745,0.960919,1.068764e-37,False
1,Amambay,2242,7.994648,7.0,4.909804,0.962477,1.138100e-23,False
2,Asunción,4035,10.727385,12.0,5.055667,0.932983,2.910017e-39,False
3,Caaguazú,2971,7.916863,7.0,4.850578,0.956980,8.791912e-29,False
4,Caazapá,2610,7.357088,6.0,4.763120,0.949595,4.685158e-29,False
5,Canindeyú,2532,7.379937,6.0,4.659420,0.960570,1.073279e-25,False
6,Central,9153,9.506938,11.0,4.874987,0.952369,2.569165e-47,False
7,Concepción,1819,7.382628,6.0,4.791453,0.954468,2.236251e-23,False
8,Cordillera,2718,8.139809,7.0,4.653424,0.962104,4.198041e-26,False
9,Guairá,2235,8.039374,7.0,4.958852,0.952572,2.536484e-26,False



Departamentos que NO cumplen normalidad: 16 / 16

Levene: W=12.963; p=3.343e-33


Se verifico por medio de la prueba de Shapiro-Wilk la No normalidad de los datos de cada departamento, es visible en la tabla de arriba que Media != Mediana. Ademas realizando la prueba de Levene se verifica que las varianzas no son homogeneas (al menos una difiere).
Ambas pruebas con un nivel de significancia α = 5%.
Por lo tanto se descarta la prueba ANOVA de un factor, con post hoc de Turkey y se opta por la alternativa no parametrica Kruskal-Wallis, con post hoc de Dunn.

In [4]:
grupos = {name:g["añoest"].to_numpy() for name,g in datos.groupby("NOMBRE_DPTO")}
kw = stats.kruskal(*grupos.values())

# k = 16 departamentos, n = total de observaciones
k, n = len(grupos), sum(map(len, grupos.values()))

# ε² mide proporcion de la variablilidad en los rangos explicada por departamento
epsilon2 = max(0, (kw.statistic-k+1)/(n-k))
print(f"Kruskal-Wallis H({k-1})={kw.statistic:.3f}; p={kw.pvalue:.4g}; epsilon²={epsilon2:.4f}; n={n:,}")

if kw.pvalue < 0.05:
    # Dunn trabaja con un DataFrame largo: una fila por observación
    df_long = datos[["añoest", "NOMBRE_DPTO"]].dropna().rename(
        columns={"añoest": "y", "NOMBRE_DPTO": "grupo"}
    )
    # p-valores ajustados por Benjamini–Hochberg (FDR).
    dunn = sp.posthoc_dunn(
        df_long,
        val_col="y",
        group_col="grupo",
        p_adjust="fdr_bh"
    )
    # La matriz viene con índices y columnas = nombres de departamento.
    # La convertimos a formato largo (una fila por par) para reportar.
    posthoc = []
    for i, a in enumerate(dunn.index):
        for b in dunn.columns[i+1:]:
            p_adj = dunn.loc[a, b]
            med_a = np.median(grupos[a])
            med_b = np.median(grupos[b])
            posthoc.append([
                a, b,
                len(grupos[a]), len(grupos[b]),
                med_a, med_b,
                med_b - med_a,       # diferencia de medianas (B - A)
                p_adj
            ])

    posthoc_df = pd.DataFrame(posthoc, columns=[
        "Departamento A", "Departamento B",
        "n A", "n B",
        "Mediana A", "Mediana B",
        "Dif. medianas (B-A)",
        "p ajustado (Dunn, BH)"
    ]).sort_values("p ajustado (Dunn, BH)")

    display(posthoc_df.head(15))
    posthoc_df.to_csv("posthoc_dunn_departamentos.csv", index=False)
else:
    print("Kruskal-Wallis no significativo: no se ejecuta post hoc de Dunn.")

Kruskal-Wallis H(15)=1997.414; p=0; epsilon²=0.0382; n=51,861


,Departamento A,Departamento B,n A,n B,Mediana A,Mediana B,Dif. medianas (B-A),"p ajustado (Dunn, BH)"
36,Asunción,Itapúa,4035,4252,12.0,6.0,-6.0,1.249768e-187
30,Asunción,Caazapá,4035,2610,12.0,6.0,-6.0,2.545560e-160
40,Asunción,San Pedro,4035,3206,12.0,6.0,-6.0,2.545560e-160
31,Asunción,Canindeyú,4035,2532,12.0,6.0,-6.0,1.381006e-150
1,Alto Paraná,Asunción,6113,4035,8.0,12.0,4.0,1.473656e-124
33,Asunción,Concepción,4035,1819,12.0,6.0,-6.0,3.012061e-123
29,Asunción,Caaguazú,4035,2971,12.0,7.0,-5.0,1.488316e-117
78,Central,Itapúa,9153,4252,11.0,6.0,-5.0,3.653771e-108
38,Asunción,Paraguarí,4035,2006,12.0,6.0,-6.0,9.370164e-94
34,Asunción,Cordillera,4035,2718,12.0,7.0,-5.0,1.704471e-93


Se uso el estadistico H, con 15 grados de libertad y se obtuvo el valor de p≈0 < 0,05 por tanto se rechaza Ho y se concluye que al menos una difiere.

##**Contraste 2: Permanencia educativa por departamentos**

**Prueba de Supuestos**

In [5]:
# Prueba de frecuencias esperadas suficientes.
# 1. Crear tabla de contingencia de ejemplo
tabla_contingencia = pd.crosstab(
    datos['DPTO'], datos['asiste_actualmente']
)

# 2. Ejecutar Chi-cuadrado y extraer las frecuencias esperadas
chi2, p_valor, dof, esperados = chi2_contingency(tabla_contingencia)

# Convertir la matriz de esperados a un DataFrame para visualizarlo claramente
df_esperados = pd.DataFrame(
    esperados,
    index=tabla_contingencia.index,
    columns=tabla_contingencia.columns,
)

print("--- Frecuencias Esperadas ---")
print(df_esperados)
print("\n" + "=" * 50 + "\n")

# 3. Validación de Supuestos
total_casillas = esperados.size
casillas_menor_1 = np.sum(esperados < 1)
casillas_menor_5 = np.sum(esperados < 5)
porcentaje_mayor_igual_5 = ((total_casillas - casillas_menor_5) / total_casillas) * 100

print(f"Total de casillas: {total_casillas}")
print(f"Casillas con esperados < 1: {casillas_menor_1}")
print(f"Casillas con esperados < 5: {casillas_menor_5} ({100 - porcentaje_mayor_igual_5:.1f}%)")
print(f"Porcentaje de casillas con esperados >= 5: {porcentaje_mayor_igual_5:.1f}%\n")

# 4. Veredicto sobre el supuesto
if casillas_menor_1 == 0 and porcentaje_mayor_igual_5 >= 80:
    print(" El supuesto de frecuencias esperadas SE CUMPLE.")
else:
    print(" El supuesto NO SE CUMPLE. Se recomienda:")


--- Frecuencias Esperadas ---
asiste_actualmente          0.0          1.0
DPTO                                        
0                   1661.537025  1212.462975
1                    785.674606   573.325394
2                   1358.020345   990.979655
3                   1112.894493   812.105507
4                    933.675120   681.324880
5                   1298.473263   947.526737
6                   1100.175699   802.824301
7                   1841.334525  1343.665475
8                   1027.909823   750.090177
9                    800.127781   583.872219
10                  2818.369171  2056.630829
11                  4035.904653  2945.095347
12                   606.455233   442.544767
13                  1036.581728   756.418272
14                  1215.222974   886.777026
15                   844.643561   616.356439


Total de casillas: 32
Casillas con esperados < 1: 0
Casillas con esperados < 5: 0 (0.0%)
Porcentaje de casillas con esperados >= 5: 100.0%

 El supuesto de fr

**Ejecución de la prueba**


In [6]:


# . Ejecutar la prueba de Chi-cuadrado de Independencia
chi2, p_valor, dof, esperados = chi2_contingency(tabla_contingencia)

print(f"Estadístico Chi-cuadrado (χ²): {chi2:.4f}")
print(f"P-valor (p-value): {p_valor:.4f}")
print(f"Grados de libertad: {dof}")
print("\nFrecuencias Esperadas:")
print(pd.DataFrame(esperados, index=tabla_contingencia.index, columns=tabla_contingencia.columns))


# . Tamaño del efecto: V de Cramer
n = tabla_contingencia.sum().sum()
min_dim = min(tabla_contingencia.shape) - 1
cramer_v = np.sqrt(chi2 / (n * min_dim))
print(f"\nV de Cramer (Intensidad de la asociación): {cramer_v:.4f}")


Estadístico Chi-cuadrado (χ²): 50.2283
P-valor (p-value): 0.0000
Grados de libertad: 15

Frecuencias Esperadas:
asiste_actualmente          0.0          1.0
DPTO                                        
0                   1661.537025  1212.462975
1                    785.674606   573.325394
2                   1358.020345   990.979655
3                   1112.894493   812.105507
4                    933.675120   681.324880
5                   1298.473263   947.526737
6                   1100.175699   802.824301
7                   1841.334525  1343.665475
8                   1027.909823   750.090177
9                    800.127781   583.872219
10                  2818.369171  2056.630829
11                  4035.904653  2945.095347
12                   606.455233   442.544767
13                  1036.581728   756.418272
14                  1215.222974   886.777026
15                   844.643561   616.356439

V de Cramer (Intensidad de la asociación): 0.0359


In [7]:
#Intervalo de confianza

datos['asiste_num'] = (datos['asiste_actualmente'] == 1).astype(int)

# Resumen por departamento
resumen = (
    datos.groupby('DPTO')['asiste_num']
    .agg(
        total='count',
        asisten='sum',
        proporcion='mean',
    )
    .reset_index()
)

# Calcular intervalo de confianza del 95% (Método Wilson o Agresti-Coull)
ci_lower, ci_upper = proportion_confint(
    count=resumen['asisten'],
    nobs=resumen['total'],
    alpha=0.05,
    method='wilson',
)

resumen['IC_Inferior_95%'] = ci_lower
resumen['IC_Superior_95%'] = ci_upper

# Mostrar resultados expresados en porcentaje
resumen['Proporción (%)'] = (resumen['proporcion'] * 100).round(2)
resumen['IC 95% (%)'] = [
    f"[{l*100:.2f}% - {u*100:.2f}%]"
    for l, u in zip(resumen['IC_Inferior_95%'], resumen['IC_Superior_95%'])
]

print(resumen[['DPTO', 'total', 'Proporción (%)', 'IC 95% (%)']])

    DPTO  total  Proporción (%)         IC 95% (%)
0      0   4035           29.91  [28.52% - 31.34%]
1      1   1819           34.85  [32.70% - 37.07%]
2      2   3206           32.13  [30.53% - 33.76%]
3      3   2718           29.84  [28.15% - 31.59%]
4      4   2235           28.72  [26.89% - 30.64%]
5      5   2971           33.39  [31.72% - 35.11%]
6      6   2610           31.65  [29.89% - 33.46%]
7      7   4252           31.96  [30.58% - 33.38%]
8      8   2480           32.18  [30.37% - 34.04%]
9      9   2006           29.36  [27.41% - 31.39%]
10    10   6113           32.95  [31.78% - 34.13%]
11    11   9153           31.14  [30.20% - 32.09%]
12    12   1637           23.76  [21.76% - 25.88%]
13    13   2242           35.28  [33.33% - 37.28%]
14    14   2532           34.20  [32.38% - 36.07%]
15    15   1852           32.61  [30.52% - 34.78%]


In [8]:
#Comparacion con bootstrap del departamento 12

# 1. Definir los datos de un departamento específico (Ejemplo: DPTO 12)
dpto = 12
n_total = 1637
prop_observada = 0.2376
exitos = int(n_total * prop_observada)

# Recrear el arreglo de datos (1 = asiste, 0 = no asiste)
datos1 = np.array([1] * exitos + [0] * (n_total - exitos))

# 2. Intervalo de Confianza Clásico (Agresti-Coull / Wilson)
ic_clasico_inf, ic_clasico_sup = proportion_confint(
    count=exitos, nobs=n_total, alpha=0.05, method='wilson'
)

# 3. Intervalo de Confianza Bootstrap (Percentil)
np.random.seed(42) # Semilla para resultados consistentes
n_bootstrap = 10000
boot_means = np.empty(n_bootstrap)

for i in range(n_bootstrap):
    boot_sample = np.random.choice(datos1, size=n_total, replace=True)
    boot_means[i] = np.mean(boot_sample)

ic_boot_inf = np.percentile(boot_means, 2.5)
ic_boot_sup = np.percentile(boot_means, 97.5)

# 4. Mostrar los resultados
print(f"--- Resultados para el Departamento {dpto} ---")
print(f"Proporción observada: {prop_observada*100:.2f}%")
print(f"IC Clásico (95%): [{ic_clasico_inf*100:.2f}% - {ic_clasico_sup*100:.2f}%]")
print(f"IC Bootstrap (95%): [{ic_boot_inf*100:.2f}% - {ic_boot_sup*100:.2f}%]")

--- Resultados para el Departamento 12 ---
Proporción observada: 23.76%
IC Clásico (95%): [21.70% - 25.82%]
IC Bootstrap (95%): [21.69% - 25.72%]


Conclusión: Se rechaza la hipótesis nula, es decir, existe evidencia suficiente para afirmar que la asistencia educativa depende del departamento de residencia.

## **Contraste tres**

**Verificacion de supuestos**

In [9]:
#Prueba de normalidad con alpha=0.05
# 3. Prueba de Normalidad (Shapiro-Wilk)
stat_e, p_e = stats.shapiro(datos['P02'])
stat_a, p_a = stats.shapiro(datos['añoest'])
print(f"Shapiro-Wilk (Edad): p-value = {p_e:.4f}")
print(f"Shapiro-Wilk (Años de estudio): p-value = {p_a:.4f}\n")

Shapiro-Wilk (Edad): p-value = 0.0000
Shapiro-Wilk (Años de estudio): p-value = 0.0000



/home/hugo/.venvs/data-science/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:579: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 51861.
  res = hypotest_fun_out(*samples, **kwds)


Como p_a y p_e son ambos menores a 0.05 se decanta por usar Spearman.

**Ejecución de la prueba**


In [10]:
coef, p_valor = stats.spearmanr(datos['P02'], datos['añoest'])
print(f"Coeficiente de Correlación: {coef:.4f}")
print(f"P-valor (p-value): {p_valor:.4f}")

Coeficiente de Correlación: 0.1801
P-valor (p-value): 0.0000


## Contraste 4: Nivel educativo y su relación con departamentos.

In [11]:
# Filtrar filas con valores válidos en ambas variables
datos_c4 = datos[["nivel_educ_simplificado", "NOMBRE_DPTO"]].dropna().copy()

# Tabla de contingencia 16 x k
tabla = pd.crosstab(datos_c4["NOMBRE_DPTO"], datos_c4["nivel_educ_simplificado"])
print("Tabla de contingencia:")
display(tabla)
print(f"\nDimensiones: {tabla.shape[0]} departamentos x {tabla.shape[1]} niveles educativos")


Tabla de contingencia:


nivel_educ_simplificado,Alfabetización/especial,Inicial/especial,Primaria (EEB 1-6),Secundaria/Media,Sin instrucción,Terciaria/Universitaria
NOMBRE_DPTO,,,,,,
Alto Paraná,4,278,2230,2340,156,1105
Amambay,5,95,842,848,60,392
Asunción,4,136,829,1581,57,1428
Caaguazú,2,118,1230,1038,80,503
Caazapá,1,93,1262,777,81,396
Canindeyú,1,103,1096,948,74,310
Central,11,357,2545,3827,120,2293
Concepción,2,71,829,584,59,274
Cordillera,2,97,1112,1011,55,441



Dimensiones: 16 departamentos x 6 niveles educativos


In [12]:
# Verificacion de supuestos: frecuencias esperadas
chi2, p_valor, dof, esperados = chi2_contingency(tabla)

df_esperados = pd.DataFrame(esperados, index=tabla.index, columns=tabla.columns)

total_casillas = esperados.size
casillas_menor_1 = np.sum(esperados < 1)
casillas_menor_5 = np.sum(esperados < 5)
porcentaje_mayor_igual_5 = ((total_casillas - casillas_menor_5) / total_casillas) * 100

print("Verificación de supuestos")
print(f"Total de casillas: {total_casillas}")
print(f"Casillas con esperados < 1: {casillas_menor_1}")
print(f"Casillas con esperados < 5: {casillas_menor_5} ({100 - porcentaje_mayor_igual_5:.1f}%)")
print(f"Porcentaje de casillas con esperados >= 5: {porcentaje_mayor_igual_5:.1f}%")

if casillas_menor_1 == 0 and porcentaje_mayor_igual_5 >= 80:
    print("El supuesto de frecuencias esperadas SE CUMPLE.")
else:
    print("El supuesto NO SE CUMPLE.")

# Prueba de chi-cuadrado
n = tabla.sum().sum()
min_dim = min(tabla.shape) - 1
cramer_v = np.sqrt(chi2 / (n * min_dim))

print("\n     Prueba       ")
print(f"Chi-cuadrado = {chi2:.4f}")
print(f"Grados de libertad: {dof}")
print(f"p-valor: {p_valor:.4g}")
print(f"n total: {n:,}")
print(f"V de Cramér: {cramer_v:.4f}")


Verificación de supuestos
Total de casillas: 96
Casillas con esperados < 1: 0
Casillas con esperados < 5: 15 (15.6%)
Porcentaje de casillas con esperados >= 5: 84.4%
El supuesto de frecuencias esperadas SE CUMPLE.

     Prueba       
Chi-cuadrado = 2122.2152
Grados de libertad: 75
p-valor: 0
n total: 51,861
V de Cramér: 0.0905


In [13]:
fila_suma = tabla.sum(axis=1).values.reshape(-1, 1)
col_suma = tabla.sum(axis=0).values.reshape(1, -1)
esperados_aj = fila_suma @ col_suma / n

residuos_ajustados = (tabla.values - esperados_aj) / np.sqrt(
    esperados_aj * (1 - fila_suma / n) * (1 - col_suma / n)
)
res_df = pd.DataFrame(residuos_ajustados, index=tabla.index, columns=tabla.columns)

# p-valor bilateral por celda
p_celdas = 2 * (1 - stats.norm.cdf(np.abs(residuos_ajustados)))
p_celdas_df = pd.DataFrame(p_celdas, index=tabla.index, columns=tabla.columns)

# Ajuste BH sobre todas las celdas
p_flat = p_celdas.flatten()
p_flat_adj = multipletests(p_flat, method="fdr_bh")[1]
p_adj_df = pd.DataFrame(
    p_flat_adj.reshape(p_celdas.shape),
    index=tabla.index,
    columns=tabla.columns
)

print("\n    Residuos ajustados (15 celdas con mayor |residuo|)")
res_long = res_df.stack().reset_index()
res_long.columns = ["Departamento", "Nivel educativo", "Residuo ajustado"]
res_long["p ajustado (BH)"] = p_adj_df.stack().values
res_long["Exceso/Defecto"] = np.where(res_long["Residuo ajustado"] > 0,
                                       "Más de lo esperado",
                                       "Menos de lo esperado")
display(res_long.reindex(res_long["Residuo ajustado"].abs().sort_values(ascending=False).index).head(15))


    Residuos ajustados (15 celdas con mayor |residuo|)


,Departamento,Nivel educativo,Residuo ajustado,p ajustado (BH),Exceso/Defecto
17,Asunción,Terciaria/Universitaria,26.068039,0.000000e+00,Más de lo esperado
14,Asunción,Primaria (EEB 1-6),-23.261901,0.000000e+00,Menos de lo esperado
38,Central,Primaria (EEB 1-6),-21.272043,0.000000e+00,Menos de lo esperado
41,Central,Terciaria/Universitaria,14.150183,0.000000e+00,Más de lo esperado
39,Central,Secundaria/Media,11.845830,0.000000e+00,Más de lo esperado
26,Caazapá,Primaria (EEB 1-6),11.663290,0.000000e+00,Más de lo esperado
62,Itapúa,Primaria (EEB 1-6),9.921345,0.000000e+00,Más de lo esperado
35,Canindeyú,Terciaria/Universitaria,-9.686606,0.000000e+00,Menos de lo esperado
86,San Pedro,Primaria (EEB 1-6),9.610360,0.000000e+00,Más de lo esperado
65,Itapúa,Terciaria/Universitaria,-7.812000,5.329071e-14,Menos de lo esperado


In [14]:
# IC por bootstrap para proporciones de un nivel específico
# Ejemplo: proporción de "Terciaria/Universitaria" por departamento
np.random.seed(42)
nivel_objetivo = "Terciaria/Universitaria"

def ic_bootstrap_prop(vals, nivel, n_boot=2000, alpha=0.05):
    """IC bootstrap para la proporción de 'nivel' en vals."""
    props = np.array([
        np.mean(np.random.choice(vals, size=len(vals), replace=True) == nivel)
        for _ in range(n_boot)
    ])
    return np.percentile(props, [100*alpha/2, 100*(1-alpha/2)])

# Comparación para dos departamentos: Asunción y Caazapá
for dpto in ["Asunción", "Caazapá"]:
    vals = datos_c4.loc[datos_c4["NOMBRE_DPTO"] == dpto, "nivel_educ_simplificado"].values
    prop = np.mean(vals == nivel_objetivo)
    ic_boot = ic_bootstrap_prop(vals, nivel_objetivo)
    # IC clásico de Wilson
    k = np.sum(vals == nivel_objetivo)
    ic_wilson = proportion_confint(k, len(vals), alpha=0.05, method="wilson")
    print(f"\n{dpto} - proporción {nivel_objetivo}:")
    print(f"  Proporción: {prop*100:.2f}%")
    print(f"  IC 95% bootstrap: [{ic_boot[0]*100:.2f}%, {ic_boot[1]*100:.2f}%]")
    print(f"  IC 95% Wilson:    [{ic_wilson[0]*100:.2f}%, {ic_wilson[1]*100:.2f}%]")


Asunción - proporción Terciaria/Universitaria:
  Proporción: 35.39%
  IC 95% bootstrap: [33.90%, 36.88%]
  IC 95% Wilson:    [33.93%, 36.88%]

Caazapá - proporción Terciaria/Universitaria:
  Proporción: 15.17%
  IC 95% bootstrap: [13.83%, 16.55%]
  IC 95% Wilson:    [13.85%, 16.60%]
